In [6]:
!pip install casadi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 11.6 MB/s eta 0:00:00


In [2]:
import numpy as np
from gekko import GEKKO
import matplotlib.pyplot as plt

# 1. Initialize Model
m = GEKKO(remote=False) # 'remote=False' solves locally
T_horizon = 100         # Time horizon (years)
steps = 101             # Number of time steps
m.time = np.linspace(0, T_horizon, steps)

# 2. Parameters (Constants)
# Values taken from the provided text [cite: 519-531]
rho = 0.014      # Discount rate
alpha = 0.3      # Capital share
delta = 0.05     # Depreciation
psi = 0.5        # Emissions per unit fossil fuel
gamma = 0.003    # Natural carbon decay
A_val = 11.9762  # Total Factor Productivity (Initial)

# Placeholders (You must update these!)
d = 504.0        # Cost of Fossils (e.g. $/tC)
c = 100.0        # Cost of Renewables
b_cost = 150.0   # Cost of Nuclear
e_cost = 1.0     # Cost of Adaptation
B_budget = 2000  # CARBON BUDGET (Constraint)
eta = 0.05       # Effectiveness of CDR

# CDR Parameters (Lines 70 in your image)
a1 = 0.01 # Investment from E
a2 = 0.001 # Degradation by P
a3 = 0.01 # Crowding out by R
a4 = 0.01 # Crowding out by N
a5 = 0.05 # Depreciation of M

# 3. Variables (Controls & States)
# We set lower bounds (lb=0) to prevent negative values

# Controls
C = m.MV(value=50, lb=0.1, name='Consumption') # Consumption
E = m.MV(value=10, lb=0, name='Fossils')       # Fossils
R = m.MV(value=1, lb=0, name='Renewables')     # Renewables
N = m.MV(value=1, lb=0, name='Nuclear')        # Nuclear
D = m.MV(value=0, lb=0, name='Adaptation')     # Adaptation Spending

# Allow the optimizer to change these controls
for var in [C, E, R, N, D]:
    var.STATUS = 1
    var.DCOST = 0

# States (with Initial Conditions from text)
K = m.Var(value=200, lb=0, name='Capital')     # Capital Stock
P = m.Var(value=826, lb=0, name='Pollution')   # Pollution Stock
M = m.Var(value=0, lb=0, name='CDR_Capacity')  # CDR Stock

# 4. Equations (Dynamics)

# Production Function (Cobb-Douglas approximation based on text)
# F(K,E,R) roughly K^alpha * Energy^beta... simplifying for structure:
# Let's assume a simplified production function for the code structure:
# Y = A * K^alpha * (E + R + N)^(1-alpha)
Production = A_val * (K**alpha) * ((E + R + N)**(1 - alpha))

# Change in Capital (Line 67-68)
# dK/dt = Production - Costs - Consumption - Depreciation
m.Equation(K.dt() == Production - d*E - c*R - b_cost*N - e_cost*D - C - delta*K)

# Change in Pollution (Line 69)
# dP/dt = Emissions - Decay - Removal
m.Equation(P.dt() == psi*E - gamma*P - eta*M)

# Change in CDR Stock (Line 70)
# dM/dt = Investment - Degradation - Crowding Out - Depreciation
m.Equation(M.dt() == a1*E - a2*P - a3*R - a4*N - a5*M)

# 5. Constraints
# Carbon Budget Constraint (Line 71)
m.Equation(P <= B_budget)

# 6. Objective Function
# Maximize sum of discounted utility: Integral( e^(-rho*t) * ln(C) )
# We use ln(C) for Utility U(C)
utility = m.log(C)
discount_factor = m.Param(value=np.exp(-rho * m.time))

# Corrected objective: GEKKO's IMODE=6 implicitly integrates the expression for Maximize/Minimize
m.Maximize(utility * discount_factor)
# Note: The previous line 'm.Maximize(utility * discount_factor * m.time[-1]/steps)' was incorrect
#       as m.time[-1]/steps is a scalar and not part of the integral formulation within GEKKO for IMODE=6.

# 7. Solve
m.options.IMODE = 6  # Dynamic Optimization
m.options.NODES = 3  # Collocation nodes
m.solve(disp=True)

# 8. Plot Results
plt.figure(figsize=(10,8))
plt.subplot(3,1,1)
plt.plot(m.time, K.value, label='Capital (K)')
plt.legend()
plt.subplot(3,1,2)
plt.plot(m.time, P.value, label='Pollution (P)')
plt.plot(m.time, [B_budget]*len(m.time), 'r--', label='Budget')
plt.legend()
plt.subplot(3,1,3)
plt.plot(m.time, E.value, label='Fossils (E)')
plt.plot(m.time, R.value, label='Renewables (R)')
plt.plot(m.time, M.value, label='CDR Capacity (M)')
plt.legend()
plt.xlabel('Time')
plt.show()

ImportError: No solution or server unreachable.
  Show errors with m.solve(disp=True).
  Try local solve with m=GEKKO(remote=False).

In [ ]:
import numpy as np
from gekko import GEKKO
import matplotlib.pyplot as plt

# --- 1. Model init
m = GEKKO(remote=False) # Changed to remote=False
T = 100.0
N = 101
m.time = np.linspace(0, T, N)

# --- 2. Parameters (as GEKKO Params)
rho = 0.014
alpha = 0.3
delta = 0.05
psi = 0.5
gamma = 0.003
A_val = 11.9762

d = 504.0
c = 100.0
b_cost = 150.0
e_cost = 1.0

B_budget = 10000.0   # CARBON BUDGET (Constraint) - Temporarily increased for testing infeasibility
beta = 1e-3         # soft penalty weight for (P-B)^2 ; increase to enforce budget more tightly

# CDR cost param (quadratic cost)
a_cdr = 0.1   # tune: larger -> more expensive CDR

# CDR dynamics params (accumulation)
a1 = 0.01  # investment via E (kept for structure if you want)
a2 = 0.001
a3 = 0.01
a4 = 0.01
a5 = 0.05   # decay of capacity

# Wrap constants as GEKko params
A = m.Param(value=A_val)
RHO = m.Param(value=rho)
ALPHA = m.Param(value=alpha)
DELTA = m.Param(value=delta)
PSI = m.Param(value=psi)
GAMMA = m.Param(value=gamma)

D_COST = m.Param(value=d)
C_COST = m.Param(value=c)
B_COST = m.Param(value=b_cost)
E_COST = m.Param(value=e_cost)

BUDGET = m.Param(value=B_budget)
BETA = m.Param(value=beta)
A_CDR = m.Param(value=a_cdr)

# --- 3. Controls (MVs)
C = m.MV(value=50.0, lb=1e-3, ub=1e5, name='Consumption')
E = m.MV(value=10.0, lb=1e-6, ub=1e4, name='Fossils')
R = m.MV(value=1.0, lb=1e-6, ub=1e4, name='Renewables')
Nuc = m.MV(value=1.0, lb=1e-6, ub=1e4, name='Nuclear')
D = m.MV(value=0.0, lb=0.0, ub=1e4, name='Adaptation')

# New: CDR deployment flow (control) - units: GtC/yr
Mflow = m.MV(value=1e-6, lb=1e-6, ub=100.0, name='CDR_flow') # Changed initial value and lb

# Let the optimizer change them
for mv in [C, E, R, Nuc, D, Mflow]:
    mv.STATUS = 1
    mv.DCOST = 0.0
    # optional smoothing:
    mv.DMAX = 50.0   # max change per time step (tune or remove if not wanted)

# --- 4. States (Vars)
K = m.Var(value=200.0, lb=1e-6, name='Capital')
P = m.Var(value=826.0, lb=0.0, name='Pollution')
Mcap = m.Var(value=1e-6, lb=1e-6, name='CDR_capacity')  # Changed initial value and lb

# Accumulator for integral objective: dJ/dt = e^{-rho t} * ln(C)
J = m.Var(value=0.0, name='AccUtility')

# --- 5. Intermediates (production)
# Use m.Intermediate for algebraic expressions (clearer & avoids broadcasting issues)
Prod = m.Intermediate( A * (K**ALPHA) * ((E + R + Nuc)**(1.0 - ALPHA)) )

# Quadratic cost of deploying removal flow (consumes goods)
h_M = m.Intermediate( 0.5 * A_CDR * Mflow**2 )

# Instantaneous penalty for exceeding budget (soft)
penalty = m.Intermediate(BETA * (P - BUDGET)**2)

# Create an intermediate for log(C) to help parser
log_C = m.Intermediate(m.log(C))

# Create a time-dependent discount factor as a parameter array
discount_values = np.exp(-rho * m.time)
discount = m.Param(value=discount_values)

# --- 6. Differential equations (dynamics)
m.Equation( K.dt() == Prod - D_COST*E - C_COST*R - B_COST*Nuc - E_COST*D - C - DELTA*K - h_M )
m.Equation( P.dt() == PSI*E - GAMMA*P - Mflow )   # Mflow directly removes atmospheric carbon
m.Equation( Mcap.dt() == a1*E - a2*P - a3*R - a4*Nuc - a5*Mcap )  # optional capacity dynamics
m.Equation( J.dt() == discount * log_C )   # Use log_C intermediate

# --- 7. Objective: maximize integral of e^{-rho t} ln(C) minus penalties/costs
# We'll minimize the negative utility + penalty integrated over time by adding penalty into the integrand
# We accomplish this by letting the optimizer maximize final J but adjusting J equation to include penalties.
# Simpler: add objective terms directly across time using m.Obj (GEKKO sums over time points).
# GEKKO allows array expressions in m.Obj; minimize negative of instantaneous integrand including penalty.

# instantaneous objective (we want to maximize): inst = discount * log_C - penalty   # Use log_C intermediate
inst = m.Intermediate( discount * log_C - penalty )  # Re-enabled objective
m.Obj( -inst )   # GEKKO sums Obj over all time points by default

# Optionally, add small regularization to avoid chattering (penalize large control changes)
# reg = 1e-4
# m.Obj( reg*(E.dt()**2 + R.dt()**2 + Mflow.dt()**2) ) # Removed due to AttributeError: dt

# --- 8. Solver options
m.options.IMODE = 6    # dynamic optimization
m.options.NODES = 3
m.options.SOLVER = 3   # IPOPT
m.options.MAX_ITER = 500
m.options.RTOL = 1e-6
# m.options.DIAGNOSTIC = 1 # Add diagnostic output - REMOVED
# m.options.ATOL = 1e-6 # Removed as ATOL is not a recognized property

# --- 9. Solve
m.solve(disp=True)

# --- 10. Plot results
t = m.time
plt.figure(figsize=(10,9))
plt.subplot(4,1,1)
plt.plot(t, K.value, label='K')
plt.legend()
plt.subplot(4,1,2)
plt.plot(t, P.value, label='P (pollution)')
plt.axhline(B_budget, color='r', linestyle='--', label='Budget B')
plt.legend()
plt.subplot(4,1,3)
plt.plot(t, E.value, label='E (fossils)')
plt.plot(t, R.value, label='R (renewables)')
plt.plot(t, Nuc.value, label='Nuclear')
plt.legend()
plt.subplot(4,1,4)
plt.plot(t, Mflow.value, label='CDR flow (Mflow)')
plt.plot(t, Mcap.value, label='CDR capacity (Mcap)')
plt.legend()
plt.xlabel('Time')
plt.show()